# FashionIQ · Exp1 reference query token PFM
Bật GPU và Internet, gắn FashionIQ đã giải nén. Mặc định Run All huấn luyện PFM ở đầu vào Q-Former reference. Target, mean pooling và contrastive loss dùng code Exp1. Bật baseline trong RUNS để chạy đối chứng.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import urllib.request

RUNS = ["reference_query_flow"]  # ["baseline"] hoặc ["reference_query_flow", "baseline"]
DATA_ROOT = None  # Hoặc Path("/kaggle/input/.../fashionIQ")
CHECKPOINT = None  # Hoặc Path("/kaggle/input/.../blip2_pretrained_vitL.pth")
SEED = 1000
WORK = Path("/kaggle/working")
EXPERIMENT_SHA = "b36607df9d7813167df9cf2ef1d1de605938f1f2"
BACKBONE_SHA = "c1c7f6b62fe6d556b9554b6c7cae111f71f67917"
EXPERIMENT = WORK / "exp_no_negative_source"
BACKBONE = WORK / "qure_main_backbone"
CODE = WORK / "exp1_reference_query_code"
CODE.mkdir(exist_ok=True)
if not RUNS or not set(RUNS) <= {"reference_query_flow", "baseline"}:
    raise ValueError("RUNS accepts reference_query_flow and baseline")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle settings.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary",
                "transformers==4.40.2", "timm==0.9.16", "fairscale", "omegaconf", "iopath"], check=True)

def checkout(repo, sha, destination):
    if not (destination / ".git").is_dir():
        destination.mkdir(exist_ok=True)
        subprocess.run(["git", "init", "-q", str(destination)], check=True)
    probe = subprocess.run(["git", "-C", str(destination), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if probe.stdout.strip() != sha:
        subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", repo, sha], check=True)
        subprocess.run(["git", "-C", str(destination), "checkout", "--detach", sha], check=True)
    print(destination.name, subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip())

checkout("https://github.com/MaiLinhLinh/Qure-Lora.git", EXPERIMENT_SHA, EXPERIMENT)
checkout("https://github.com/fast2104/QuRe.git", BACKBONE_SHA, BACKBONE)

In [ ]:
import shutil
inputs = Path("/kaggle/input")
if DATA_ROOT is None:
    matches = [p.parent.parent for p in inputs.rglob("cap.dress.train.json")
               if (p.parent.parent / "images").is_dir() and (p.parent.parent / "image_splits").is_dir()]
    if len(matches) != 1:
        raise ValueError(f"Set DATA_ROOT to the extracted FashionIQ directory. Candidates: {matches}")
    DATA_ROOT = matches[0]
DATA_ROOT = Path(DATA_ROOT)
for category in ("dress", "shirt", "toptee"):
    for split in ("train", "val"):
        annotations = DATA_ROOT / "captions" / f"cap.{category}.{split}.json"
        sample = json.loads(annotations.read_text())[0]
        for key in ("candidate", "target"):
            image = DATA_ROOT / "images" / (sample[key] + ".png")
            if not image.is_file():
                raise FileNotFoundError(image)
    assert (DATA_ROOT / "image_splits" / f"split.{category}.val.json").is_file()

name = "blip2_pretrained_vitL.pth"
if CHECKPOINT is None:
    CHECKPOINT = next(inputs.rglob(name), WORK / name)
CHECKPOINT = Path(CHECKPOINT).resolve()
if not CHECKPOINT.is_file():
    if CHECKPOINT != WORK / name:
        raise FileNotFoundError(CHECKPOINT)
    base_url = "https://storage.googleapis.com/sfr-vision-language-research/LAVIS/models/BLIP2/"
    temporary = WORK / (name + ".part")
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(temporary), base_url + name], check=True)
    temporary.replace(CHECKPOINT)
if CHECKPOINT.name != name:
    raise ValueError(f"Expected checkpoint filename {name}")
os.environ["QURe_CHECKPOINT_ROOT"] = str(CHECKPOINT.parent)
print("FashionIQ:", DATA_ROOT, "| BLIP-2:", CHECKPOINT)

## Backbone và trainer
Chỉ nạp các module LAVIS cần cho retrieval trong checkout riêng của notebook. Cả hai chế độ gọi cùng model, tokenizer, tiền xử lý và evaluator.

In [ ]:
def original(rel):
    return subprocess.check_output(["git", "-C", str(BACKBONE), "show", f"HEAD:{rel}"], text=True)

model_init = "lavis/models/__init__.py"
source = original(model_init)
loaders = source[source.index("def load_model("):source.index("class ModelZoo")]
imports = """import logging
import torch
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models.base_model import BaseModel
from lavis.processors import BaseProcessor
from lavis.models.blip2_models.blip2_qure import Blip2QuRe
"""
(BACKBONE / model_init).write_text(imports + "\n" + loaders)
(BACKBONE / "lavis/processors/__init__.py").write_text(
    "from lavis.processors.base_processor import BaseProcessor\nfrom lavis.processors import blip_processors\n")
(BACKBONE / "data/__init__.py").write_text("# Retrieval runner imports only data.utils.\n")
yaml_path = "lavis/configs/models/blip2/blip2_pretrain_vitL.yaml"
import re
yaml_text, count = re.subn(r"(?m)^  pretrained:.*$", lambda _: "  pretrained: " + json.dumps(str(CHECKPOINT)), original(yaml_path))
assert count == 1
(BACKBONE / yaml_path).write_text(yaml_text)
for filename in ["data.py", "model.py"]:
    shutil.copyfile(EXPERIMENT / "Code_KT" / filename, CODE / filename)

## Query token theo ảnh reference
Bốn lớp additive coupling biến đổi bản sao trainable của query token pretrained. Điều kiện là đặc trưng ViT của ảnh reference. Lớp cuối của từng coupling bắt đầu bằng 0.

In [ ]:
flow_source = '''import torch
from torch import nn


class ReferenceQueryFlow(nn.Module):
    def __init__(self, queries, vision_width, hidden=64):
        super().__init__()
        self.seed_queries = nn.Parameter(queries.detach().float().clone())
        half = queries.shape[-1] // 2
        self.condition = nn.Sequential(nn.LayerNorm(vision_width), nn.Linear(vision_width, hidden), nn.SiLU())
        self.shifts = nn.ModuleList([
            nn.Sequential(nn.Linear(half + hidden, hidden), nn.SiLU(), nn.Linear(hidden, half))
            for _ in range(4)
        ])
        for shift in self.shifts:
            nn.init.zeros_(shift[-1].weight)
            nn.init.zeros_(shift[-1].bias)

    def forward(self, visual):
        with torch.autocast(device_type=visual.device.type, enabled=False):
            tokens = self.seed_queries.expand(visual.shape[0], -1, -1)
            context = self.condition(visual.float().mean(dim=1))
            context = context[:, None, :].expand(-1, tokens.shape[1], -1)
            left, right = tokens.chunk(2, dim=-1)
            for i, shift in enumerate(self.shifts):
                if i % 2 == 0:
                    right = right + shift(torch.cat([left, context], dim=-1))
                else:
                    left = left + shift(torch.cat([right, context], dim=-1))
            return torch.cat([left, right], dim=-1)
'''
(CODE / "reference_query_flow.py").write_text(flow_source)

In [ ]:
def replace_once(source, old, new):
    if source.count(old) != 1:
        raise RuntimeError(f"Unexpected source near {old!r}")
    return source.replace(old, new)

# Dedicated model file; the baseline trainer imports the upstream model.
model_source = (EXPERIMENT / "Code_KT/model.py").read_text()
model_source = replace_once(model_source, "import torch.nn as nn",
                            "import torch.nn as nn\nfrom reference_query_flow import ReferenceQueryFlow")
model_source = replace_once(model_source, "        self.encoder_width = base.Qformer.config.encoder_width",
                            "        self.encoder_width = base.Qformer.config.encoder_width\n"
                            "        self.ref_query_flow = ReferenceQueryFlow(base.query_tokens, self.encoder_width)")
model_source = replace_once(model_source, "        batch_size = reference_images.shape[0]\n        queries = self._queries(batch_size)",
                            "        batch_size = reference_images.shape[0]\n        queries = self.ref_query_flow(visual)")
compile(model_source, "reference_model.py", "exec")
(CODE / "reference_model.py").write_text(model_source)
trainer = (EXPERIMENT / "Code_KT/train.py").read_text()
(CODE / "train.py").write_text(trainer)
flow_trainer = replace_once(trainer, "from model import ", "from reference_model import ")
(CODE / "train_reference.py").write_text(flow_trainer)

## Kiểm tra trước khi train
Kiểm tra token lúc khởi tạo, gradient của flow và sự khác nhau giữa hai ảnh sau một bước cập nhật.

In [ ]:
smoke = """
import torch
from reference_query_flow import ReferenceQueryFlow
torch.manual_seed(1000)
seed = torch.randn(1, 32, 768)
flow = ReferenceQueryFlow(seed, 1024)
visual = torch.randn(2, 17, 1024)
queries = flow(visual)
torch.testing.assert_close(queries, seed.expand_as(queries))
(queries * torch.randn_like(queries)).mean().backward()
grad = flow.shifts[0][-1].weight.grad
assert grad is not None and torch.isfinite(grad).all() and grad.abs().sum() > 0
with torch.no_grad():
    flow.shifts[0][-1].weight.add_(grad, alpha=-1.0)
    updated = flow(visual)
    assert not torch.equal(updated[0], updated[1])
print("PASS: initialization, gradient, image conditioning")
"""
subprocess.run([sys.executable, "-c", smoke], cwd=CODE, check=True)

In [ ]:
template = json.loads((EXPERIMENT / "Code_KT/configs/exp1.json").read_text())
for mode in RUNS:
    cfg = dict(template, seed=SEED, reference_queries=mode,
               experiment_sha=EXPERIMENT_SHA, backbone_sha=BACKBONE_SHA)
    output = WORK / f"exp1_{mode}_seed{SEED}"
    cfg["output_dir"] = str(output)
    config_path = CODE / f"config_{mode}.json"
    config_path.write_text(json.dumps(cfg, indent=2))
    args = [sys.executable, "-u", str(CODE / ("train_reference.py" if mode == "reference_query_flow" else "train.py")), "--config", str(config_path),
            "--qure-root", str(BACKBONE), "--fashioniq-root", str(DATA_ROOT), "--output-dir", str(output)]
    if (output / "last.pt").is_file():
        if json.loads((output / "config.json").read_text()) != cfg:
            raise ValueError(f"Existing checkpoint has a different config: {output}")
        args.extend(["--resume", str(output / "last.pt")])
    print(mode, output, flush=True)
    subprocess.run(args, cwd=CODE, check=True)

In [ ]:
import pandas as pd
rows = {}
for mode in ("baseline", "reference_query_flow"):
    path = WORK / f"exp1_{mode}_seed{SEED}" / "history.json"
    if path.exists():
        history = json.loads(path.read_text())
        row = next((r for r in history if r["epoch"] == template["epochs"]), None)
        if row:
            rows[mode] = {k: row[k] for k in ("average_recall_at10", "average_recall_at50", "average_recall")}
if rows:
    result = pd.DataFrame.from_dict(rows, orient="index")
    if len(rows) == 2:
        result.loc["PFM − baseline"] = result.loc["reference_query_flow"] - result.loc["baseline"]
    display(result.round(3))